# Urban 3D Perception: run on a free Colab GPU
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Upload **urban-3d-perception.zip** when asked (step 1), then your **dashcam video** (step 3).

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader || echo 'No GPU: switch the runtime to T4 GPU'

## 1. Get the project
Upload `urban-3d-perception.zip`. If the code is already on GitHub, set `REPO` instead.

In [ ]:
import os, zipfile
from google.colab import files
REPO = ""   # e.g. "https://github.com/your-username/urban-3d-perception"
os.chdir("/content")
if REPO:
    !git clone -q {REPO} urban-3d-perception
else:
    up = files.upload()
    zipfile.ZipFile(list(up)[0]).extractall("/content")
os.chdir("/content/urban-3d-perception")
print(os.listdir("."))

## 2. Install and test

In [ ]:
!pip -q install ultralytics lap transformers pyyaml
!python -m pytest -q tests

## 3. Upload a dashcam video
A 20–60 s clip with traffic ahead works best, e.g. search "dashcam" or "driving POV" on pexels.com.

In [ ]:
import shutil
os.makedirs("samples", exist_ok=True)
up = files.upload()
shutil.move(list(up)[0], "samples/drive.mp4")
print("Saved samples/drive.mp4")

## 4. Run the pipeline
If distances look too large or small, rerun with a different `--hfov` (most dashcams are 60–90°).

In [ ]:
!python scripts/run.py --video samples/drive.mp4 --max-frames 300 --stride 2

## 5. Results

In [ ]:
from IPython.display import Markdown
Markdown(open('outputs/results.md').read())

## 6. Open the web dashboard inside Colab

In [ ]:
import subprocess, time
from google.colab import output
subprocess.Popen(["python", "-m", "http.server", "-d", "docs", "8000"])
time.sleep(1)
output.serve_kernel_port_as_iframe(8000, height=1400)

## 7. Make the README GIF and download everything
The zip contains the code, the dashboard data, the GIF and the results. Upload its contents to your GitHub repo.

In [ ]:
!ffmpeg -y -loglevel error -t 8 -i outputs/annotated.mp4 -vf "fps=8,scale=900:-1:flags=lanczos,split[a][b];[a]palettegen[p];[b][p]paletteuse" docs/demo.gif
!cd /content && zip -qr urban-3d-perception-results.zip urban-3d-perception -x "*/samples/*" "*.pt" "*/__pycache__/*" "*/.pytest_cache/*"
files.download("/content/urban-3d-perception-results.zip")